# Multi-Judge Evaluation Pipeline

This notebook evaluates predictions from the GRPO model using 7 different judge metrics:
1. **Exact Match Judge** - Binary exact JSON match
2. **Structural F1 Judge** - Node and edge F1 scores
3. **Semantic Similarity Judge** - Embedding-based similarity
4. **OpenAI Issues** - GPT-5 quality assessment
5. **Hybrid Judge** - Combined structural + semantic
6. **Parse Success Judge** - JSON validity check
7. **Completeness Judge** - Coverage metrics
8. **Graph Edit Distance** - Captures structure and semantics, inverted
9. **Ontology Compliance Judge** - Captures semantics


Linter metrics judges for graph validity

10. **Unused Definitions Judge**
11. **Undefined References Judge**
12. **Duplicate Definitions Judge**
13. **Orphan Nodes Judge**
14. **Isolated Components Judge**
15. **Domain Range Validity Judge**
16. **Label Accuracy Judge**




**Input**: Predictions from `/content/drive/MyDrive/multiJudgePredictions-full/`

**Output**: Comprehensive scores saved to Google Drive

## 1. Setup and Imports

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# Install required packages
!pip install -q sentence-transformers openai scikit-learn numpy pandas tqdm tenacity

In [ ]:
import json
import os
import re
from pathlib import Path
from typing import Dict, List, Tuple, Optional, Any
from collections import defaultdict
import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from datetime import datetime

# For semantic similarity
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

import openai

print("✓ All imports successful")

✓ All imports successful


## 2. Configuration

In [ ]:
# Paths
PREDICTIONS_DIR = Path("/content/drive/MyDrive/multiJudgePredictions-full")
SAMPLES_DIR = PREDICTIONS_DIR / "samples"
OUTPUT_DIR = Path("/content/drive/MyDrive/multiJudgeScores_non_gold")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# OPENAI setup
import os
from google.colab import userdata

GEM_API_KEY = userdata.get('gemini')

OPENAI_API_KEY = userdata.get('OpenAIService')
USE_LLM_JUDGE = bool(OPENAI_API_KEY)

print(f"Predictions directory: {PREDICTIONS_DIR}")
print(f"Output directory: {OUTPUT_DIR}")
print(f"LLM Judge enabled: {USE_LLM_JUDGE}")

Predictions directory: /content/drive/MyDrive/multiJudgePredictions-full
Output directory: /content/drive/MyDrive/multiJudgeScores_non_gold
LLM Judge enabled: True


## 3. Load Embedding Model

In [ ]:
# Load sentence transformer for semantic similarity
print("Loading embedding model...")
embedding_model = SentenceTransformer('all-MiniLM-L6-v2')
print("✓ Embedding model loaded")

Loading embedding model...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

... (3 outputs left out)

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

✓ Embedding model loaded


## 4. Utility Functions

In [ ]:
import re
import json
from typing import Dict, List, Tuple, Optional, Any

def normalize_text(text: str) -> str:
    """Normalize text for comparison."""
    return re.sub(r'\s+', ' ', str(text).strip().lower())


def normalize_node_for_comparison(node: dict) -> dict:
    """
    Normalize node structure to handle both schemas:
    1. Flat: {"id": "x", "node_type": "PARTY", "name": "ACME"}
    2. Nested: {"id": "x", "type": "PARTY", "attrs": {"name": "ACME"}}

    This flattens the structure and ensures consistent field names.
    """
    if not isinstance(node, dict):
        return node

    normalized = dict(node)

    # If attrs exists, merge it to top level (don't overwrite existing)
    if 'attrs' in node and isinstance(node['attrs'], dict):
        attrs = node['attrs']
        for key, value in attrs.items():
            if key not in normalized:
                normalized[key] = value

    # Ensure both 'type' and 'node_type' are available at top level
    if 'type' in normalized and 'node_type' not in normalized:
        normalized['node_type'] = normalized['type']
    elif 'node_type' in normalized and 'type' not in normalized:
        normalized['type'] = normalized['node_type']

    return normalized


def normalize_graph_schema(graph: dict) -> dict:
    """Normalize entire graph schema to handle different formats."""
    if not isinstance(graph, dict):
        return graph

    normalized = dict(graph)

    # Normalize nodes
    if 'nodes' in graph and isinstance(graph['nodes'], list):
        normalized['nodes'] = [normalize_node_for_comparison(n) for n in graph['nodes']]

    return normalized


def extract_nodes(graph: dict) -> List[dict]:
    """Extract nodes from graph."""
    return graph.get('nodes', []) if isinstance(graph, dict) else []


def extract_edges(graph: dict) -> List[dict]:
    """Extract edges from graph."""
    return graph.get('edges', []) if isinstance(graph, dict) else []


def safe_json_parse(text: str) -> Optional[dict]:
    """Safely parse JSON from text."""
    if not isinstance(text, str):
        return None

    try:
        return json.loads(text)
    except:
        # Try to extract JSON from text
        start = text.find('{')
        end = text.rfind('}') + 1
        if start != -1 and end > start:
            try:
                return json.loads(text[start:end])
            except:
                pass
    return None


def calculate_prf1(tp: int, fp: int, fn: int) -> Tuple[float, float, float]:
    """Calculate precision, recall, F1."""
    p = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    r = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1 = (2 * p * r) / (p + r) if (p + r) > 0 else 0.0
    return p, r, f1


print("✓ Utility functions defined")

✓ Utility functions defined


## 5. Judge Implementations

In [ ]:
class Judge1_ExactMatch:
    """Judge 1: Exact Match - Binary exact JSON comparison."""

    def __init__(self):
        self.name = "exact_match"

    def score(self, pred: dict, gold: dict) -> Dict[str, Any]:
        pred_json = safe_json_parse(pred) if isinstance(pred, str) else pred
        gold_json = safe_json_parse(gold) if isinstance(gold, str) else gold

        if pred_json is None or gold_json is None:
            return {
                'score': 0.0,
                'exact_match': False,
                'valid_json': pred_json is not None
            }

        # Normalize for comparison
        pred_str = json.dumps(pred_json, sort_keys=True)
        gold_str = json.dumps(gold_json, sort_keys=True)

        exact_match = pred_str == gold_str

        return {
            'score': 1.0 if exact_match else 0.0,
            'exact_match': exact_match,
            'valid_json': True
        }


print("✓ Judge 1 (Exact Match) defined")

✓ Judge 1 (Exact Match) defined


In [ ]:
class Judge2_StructuralF1:
    """Judge 2: Structural F1 - Node and edge matching with F1 scores (FIXED VERSION)."""

    def __init__(self):
        self.name = "structural_f1"

    def _node_to_tuple(self, node: dict) -> tuple:
        """
        Convert node to comparable tuple with schema-aware attribute extraction.

        FIXED: Now checks both top-level AND attrs dict for attributes.
        """
        node_id = node.get('id', '')
        node_type = node.get('node_type', node.get('type', '')).upper()

        key_attr = ''
        attrs = node.get('attrs', {})

        for attr_name in ['name', 'term', 'text', 'title']:
            if attr_name in node and node[attr_name]:
                key_attr = normalize_text(str(node[attr_name]))
                break
            elif isinstance(attrs, dict) and attr_name in attrs and attrs[attr_name]:
                key_attr = normalize_text(str(attrs[attr_name]))
                break

        if not key_attr and node_id:
            key_attr = normalize_text(str(node_id))

        return (node_type, key_attr)

    def _edge_to_tuple(self, edge: dict, node_map: dict) -> tuple:
        """
        Convert edge to comparable tuple with fallback for unmapped nodes.

        FIXED: Better handling of multiple edge field names and fallbacks.
        """
        src = edge.get('src', edge.get('source', edge.get('from', '')))
        tgt = edge.get('tgt', edge.get('target', edge.get('to', '')))
        edge_type = edge.get('type', edge.get('edge_type', edge.get('label', ''))).upper()

        src_key = node_map.get(str(src), str(src)) if src else ''
        tgt_key = node_map.get(str(tgt), str(tgt)) if tgt else ''

        return (src_key, edge_type, tgt_key)

    def score(self, pred: dict, gold: dict) -> Dict[str, Any]:
        pred_json = safe_json_parse(pred) if isinstance(pred, str) else pred
        gold_json = safe_json_parse(gold) if isinstance(gold, str) else gold

        if pred_json is None or gold_json is None:
            return {
                'score': 0.0,
                'node_precision': 0.0,
                'node_recall': 0.0,
                'node_f1': 0.0,
                'edge_precision': 0.0,
                'edge_recall': 0.0,
                'edge_f1': 0.0,
                'valid_json': pred_json is not None
            }

        pred_json = normalize_graph_schema(pred_json)
        gold_json = normalize_graph_schema(gold_json)

        pred_nodes = extract_nodes(pred_json)
        gold_nodes = extract_nodes(gold_json)
        pred_edges = extract_edges(pred_json)
        gold_edges = extract_edges(gold_json)

        pred_node_map = {str(n.get('id', '')): self._node_to_tuple(n) for n in pred_nodes}
        gold_node_map = {str(n.get('id', '')): self._node_to_tuple(n) for n in gold_nodes}

        pred_node_set = set(pred_node_map.values())
        gold_node_set = set(gold_node_map.values())

        pred_edge_set = {self._edge_to_tuple(e, pred_node_map) for e in pred_edges}
        gold_edge_set = {self._edge_to_tuple(e, gold_node_map) for e in gold_edges}

        node_tp = len(pred_node_set & gold_node_set)
        node_fp = len(pred_node_set - gold_node_set)
        node_fn = len(gold_node_set - pred_node_set)

        node_p, node_r, node_f1 = calculate_prf1(node_tp, node_fp, node_fn)

        edge_tp = len(pred_edge_set & gold_edge_set)
        edge_fp = len(pred_edge_set - gold_edge_set)
        edge_fn = len(gold_edge_set - pred_edge_set)

        edge_p, edge_r, edge_f1 = calculate_prf1(edge_tp, edge_fp, edge_fn)

        # Overall score (average of node and edge F1)
        overall_score = (node_f1 + edge_f1) / 2.0

        return {
            'score': overall_score,
            'node_precision': node_p,
            'node_recall': node_r,
            'node_f1': node_f1,
            'edge_precision': edge_p,
            'edge_recall': edge_r,
            'edge_f1': edge_f1,
            'valid_json': True
        }
print("✓ Judge 2 (Structural F1) defined")

✓ Judge 2 (Structural F1) defined


In [ ]:
class Judge3_SemanticSimilarity:
    """Judge 3: Semantic Similarity - Embedding-based comparison (FIXED VERSION)."""

    def __init__(self, embedding_model):
        self.name = "semantic_similarity"
        self.model = embedding_model

    def _graph_to_text(self, graph: dict) -> str:
        """
        Convert graph to text representation.

        FIXED: Now extracts attributes from both top-level and attrs dict.
        """
        if not isinstance(graph, dict):
            return ""

        parts = []

        nodes = extract_nodes(graph)
        for node in nodes:
            node_type = node.get('node_type', node.get('type', ''))
            node_id = node.get('id', '')

            attr = ''
            attrs = node.get('attrs', {})
            for attr_name in ['name', 'text', 'title', 'term']:
                if attr_name in node and node[attr_name]:
                    attr = node[attr_name]
                    break
                elif isinstance(attrs, dict) and attr_name in attrs and attrs[attr_name]:
                    attr = attrs[attr_name]
                    break

            parts.append(f"{node_type} {node_id} {attr}")

        # Add edges
        edges = extract_edges(graph)
        for edge in edges:
            src = edge.get('src', edge.get('source', ''))
            tgt = edge.get('tgt', edge.get('target', ''))
            edge_type = edge.get('type', edge.get('edge_type', ''))
            parts.append(f"{src} {edge_type} {tgt}")

        return " ".join(parts)

    def score(self, pred: dict, gold: dict) -> Dict[str, Any]:
        pred_json = safe_json_parse(pred) if isinstance(pred, str) else pred
        gold_json = safe_json_parse(gold) if isinstance(gold, str) else gold

        if pred_json is None or gold_json is None:
            return {
                'score': 0.0,
                'cosine_similarity': 0.0,
                'valid_json': pred_json is not None
            }

        pred_json = normalize_graph_schema(pred_json)
        gold_json = normalize_graph_schema(gold_json)

        # Convert to text
        pred_text = self._graph_to_text(pred_json)
        gold_text = self._graph_to_text(gold_json)

        if not pred_text or not gold_text:
            return {
                'score': 0.0,
                'cosine_similarity': 0.0,
                'valid_json': True
            }

        # Compute embeddings
        pred_emb = self.model.encode([pred_text])
        gold_emb = self.model.encode([gold_text])

        # Compute cosine similarity
        from sklearn.metrics.pairwise import cosine_similarity
        similarity = float(cosine_similarity(pred_emb, gold_emb)[0][0])

        return {
            'score': similarity,
            'cosine_similarity': similarity,
            'valid_json': True
        }

In [ ]:

from dataclasses import dataclass
from typing import Dict, Any, Optional
import json, time, os, math
from tenacity import retry, stop_after_attempt, wait_exponential, retry_if_exception_type
from openai import OpenAI

@dataclass
class OpenAIJudgeConfig:
    model: str = os.getenv("OPENAI_LLM_JUDGE_MODEL", "gpt-5-thinking")
    temperature: float = float(os.getenv("OPENAI_LLM_TEMPERATURE", "0.0"))
    timeout: float = float(os.getenv("OPENAI_LLM_TIMEOUT", "60"))
    max_retries: int = int(os.getenv("OPENAI_LLM_MAX_RETRIES", "1"))

    insist_json: bool = True

class Judge4_OpenAI_Issues:
    """Judge 4: OpenAI Issues Counter - GPT-5 Thinking identifies and counts errors/issues."""
    def __init__(self, enabled: bool = False, cfg: Optional[OpenAIJudgeConfig] = None):
        self.name = "openai_issues"
        self.enabled = enabled
        self.call_count = 0
        self.success_count = 0
        self.error_count = 0
        self.cfg = cfg or OpenAIJudgeConfig()

        self._client = None
        if self.enabled:
            try:
                api_key = userdata.get("OpenAIService")
                if not api_key:
                    raise RuntimeError("OPENAI_API_KEY not set in environment.")
                self._client = OpenAI(api_key=api_key, timeout=self.cfg.timeout, max_retries=self.cfg.max_retries)
                print(f"✓ {self.name} initialized with API access (model: {self.cfg.model}, timeout={self.cfg.timeout}s)")
            except Exception as e:
                print(f"⚠ Failed to initialize OpenAI client: {e}")
                self.enabled = False
        else:
            print(f"⚠ {self.name} initialized but DISABLED (no API key or flag off)")

    def _build_prompt(self, pred: dict, gold: dict, clause_text: str, original_prompt: str) -> str:
        return (
            "You are a meticulous evaluator for legal graph extraction.\n"
            "Given a PRED graph (model prediction) and a GOLD graph (ground truth), "
            "identify concrete issues in PRED compared to GOLD, based only on the provided clause text and graphs.\n\n"
            "Return a strict JSON object with fields:\n"
            "  - score: number in [0,1] where 1=perfectly correct, 0=completely wrong\n"
            "  - issues_found: integer count of distinct issues\n"
            "  - issue_list: array of short issue strings (e.g., 'missing node TERM: Force Majeure', 'wrong edge type BETWEEN A->B')\n"
            "  - explanation: brief rationale (1-3 sentences)\n\n"
            f"CLAUSE_TEXT:\n{clause_text}\n\n"
            f"ORIGINAL_PROMPT:\n{original_prompt}\n\n"
            f"PRED_GRAPH(JSON):\n{json.dumps(pred, ensure_ascii=False)}\n\n"
            f"GOLD_GRAPH(JSON):\n{json.dumps(gold, ensure_ascii=False)}\n\n"
            "Respond with ONLY the JSON object. No prose."
        )

    @retry(stop=stop_after_attempt(1),
           wait=wait_exponential(multiplier=1.5, min=1, max=20),
           retry=retry_if_exception_type(Exception))
    def _call_openai(self, sys_prompt: str, user_prompt: str) -> str:
        resp = self._client.chat.completions.create(
            model="gpt-5",
            messages=[
                {"role": "system", "content": sys_prompt},
                {"role": "user", "content": user_prompt},
            ],
            #temperature=self.cfg.temperature,
            response_format={"type": "json_object"} if self.cfg.insist_json else None,
        )
        print(f"[{self.name}] got response: {resp}")
        text = resp.choices[0].message.content or "{}"

        try:
            print(f"OpenAI response: {resp}")
            return text
        except Exception:
            for out in getattr(resp, "output", []) or []:
                for item in getattr(out, "content", []) or []:
                    if getattr(item, "type", None) == "output_text" and getattr(item, "text", None):
                        return item.text
            return json.dumps({"score": None, "issues_found": None, "issue_list": [], "explanation": "No text output"})

    def _parse_json(self, text: str) -> Dict[str, Any]:
        try:
            data = json.loads(text)
            score = data.get("score")
            if isinstance(score, str):
                try:
                    score = float(score)
                except:
                    score = None
            if score is not None:
                score = max(0.0, min(1.0, float(score)))

            issues_found = data.get("issues_found")
            if isinstance(issues_found, str):
                try:
                    issues_found = int(issues_found)
                except:
                    issues_found = None

            issue_list = data.get("issue_list") or []
            if not isinstance(issue_list, list):
                issue_list = [str(issue_list)]

            explanation = data.get("explanation") or ""

            return {
                "score": score,
                "issues_found": issues_found if isinstance(issues_found, int) else (len(issue_list) if issue_list else None),
                "issue_list": [str(x) for x in issue_list][:20],
                "explanation": str(explanation)[:1000]
            }
        except Exception as e:
            return {
                "score": None,
                "issues_found": None,
                "issue_list": [],
                "explanation": f"JSON parse error: {e}",
            }

    def score(self, pred: dict, gold: dict, clause_text: str = "", original_prompt: str = "") -> Dict[str, Any]:
        self.call_count += 1
        if not self.enabled:
            if self.call_count == 1:
                print(f"[{self.name}] Disabled: set OPENAI_API_KEY to enable.")
            return {'score': None, 'issues_found': None, 'issue_list': [], 'explanation': 'LLM judge disabled', 'enabled': False}

        try:
            sys_msg = "You are an expert evaluator for legal graph extraction."
            user_msg = self._build_prompt(pred, gold, clause_text, original_prompt)
            print(f"[{self.name}] sending call: {user_msg}")
            raw = self._call_openai(sys_msg, user_msg)
            print(f"[{self.name}] got response: {raw}")
            parsed = self._parse_json(raw)
            self.success_count += 1
            return {**parsed, 'enabled': True}
        except Exception as e:
            self.error_count += 1
            print(f"[{self.name}] ❌ Error: {type(e).__name__}: {str(e)}")
            return {'score': None, 'issues_found': None, 'issue_list': [], 'explanation': f'Error: {e}', 'enabled': True}

print("✓ Judge 4 (OpenAI Issues Counter) updated for gpt-5-thinking with Responses API and retries")


✓ Judge 4 (OpenAI Issues Counter) updated for gpt-5-thinking with Responses API and retries


In [ ]:
class Judge6_Hybrid:
    """Judge 6: Hybrid - Combined structural and semantic scoring."""

    def __init__(self, structural_judge, semantic_judge, alpha: float = 0.6):
        self.name = "hybrid"
        self.structural = structural_judge
        self.semantic = semantic_judge
        self.alpha = alpha

    def score(self, pred: dict, gold: dict) -> Dict[str, Any]:
        struct_result = self.structural.score(pred, gold)
        sem_result = self.semantic.score(pred, gold)

        struct_score = struct_result.get('score', 0.0)
        sem_score = sem_result.get('score', 0.0)

        # Weighted combination
        hybrid_score = self.alpha * struct_score + (1 - self.alpha) * sem_score

        return {
            'score': hybrid_score,
            'structural_score': struct_score,
            'semantic_score': sem_score,
            'alpha': self.alpha,
            'valid_json': struct_result.get('valid_json', False)
        }


print("✓ Judge 6 (Hybrid) defined")

✓ Judge 6 (Hybrid) defined


In [ ]:
class Judge7_ParseSuccess:
    """Judge 7: Parse Success - JSON validity and structure checks."""

    def __init__(self):
        self.name = "parse_success"

    def score(self, pred: dict, gold: dict) -> Dict[str, Any]:
        pred_json = safe_json_parse(pred) if isinstance(pred, str) else pred

        if pred_json is None:
            return {
                'score': 0.0,
                'valid_json': False,
                'has_nodes': False,
                'has_edges': False,
                'has_contract_id': False
            }
        pred_json = normalize_graph_schema(pred_json)
        has_nodes = 'nodes' in pred_json and len(pred_json.get('nodes', [])) > 0
        has_edges = 'edges' in pred_json and len(pred_json.get('edges', [])) > 0
        has_contract_id = 'contract_id' in pred_json

        # Score based on presence of expected fields
        score = 0.0
        if has_contract_id:
            score += 0.2
        if has_nodes:
            score += 0.4
        if has_edges:
            score += 0.4

        return {
            'score': score,
            'valid_json': True,
            'has_nodes': has_nodes,
            'has_edges': has_edges,
            'has_contract_id': has_contract_id
        }


print("✓ Judge 7 (Parse Success) defined")

✓ Judge 7 (Parse Success) defined


In [ ]:
from typing import Dict, Any

class Judge8_Completeness:
    """Judge 8: Completeness - Coverage and recall-focused metrics."""
    def __init__(self):
        self.name = "completeness"

    def score(self, pred: dict, gold: dict) -> Dict[str, Any]:
        pred_json = safe_json_parse(pred) if isinstance(pred, str) else pred
        gold_json = safe_json_parse(gold) if isinstance(gold, str) else gold

        if pred_json is None or gold_json is None:
            return {
                'score': 0.0,
                'node_coverage': 0.0,
                'edge_coverage': 0.0,
                'type_coverage': 0.0,
                'valid_json': pred_json is not None
            }

        pred_json = normalize_graph_schema(pred_json)
        gold_json = normalize_graph_schema(gold_json)
        pred_nodes = extract_nodes(pred_json)
        gold_nodes = extract_nodes(gold_json)
        pred_edges = extract_edges(pred_json)
        gold_edges = extract_edges(gold_json)

        # Node coverage (how many gold nodes are covered)
        gold_node_ids = {n.get('id') for n in gold_nodes}
        pred_node_ids = {n.get('id') for n in pred_nodes}
        node_coverage = (len(gold_node_ids & pred_node_ids) / max(1, len(gold_node_ids))) if gold_node_ids else 0.0

        # Edge coverage (match by (source,target,type))
        def edge_key(e):
            return (
                e.get('source') or e.get('from'),
                e.get('target') or e.get('to'),
                e.get('type') or e.get('label') or e.get('relation')
            )
        gold_edge_keys = {edge_key(e) for e in gold_edges}
        pred_edge_keys = {edge_key(e) for e in pred_edges}
        edge_coverage = (len(gold_edge_keys & pred_edge_keys) / max(1, len(gold_edge_keys))) if gold_edge_keys else 0.0

        # Type coverage: fraction of gold node types present in pred
        def node_type(n):
            return n.get('type') or n.get('label') or n.get('category')
        gold_types = {node_type(n) for n in gold_nodes if node_type(n)}
        pred_types = {node_type(n) for n in pred_nodes if node_type(n)}
        type_coverage = (len(gold_types & pred_types) / max(1, len(gold_types))) if gold_types else 0.0

        score = 0.5 * node_coverage + 0.3 * edge_coverage + 0.2 * type_coverage
        return {
            'score': float(score),
            'node_coverage': float(node_coverage),
            'edge_coverage': float(edge_coverage),
            'type_coverage': float(type_coverage),
            'valid_json': True,
            'num_gold_nodes': len(gold_nodes),
            'num_pred_nodes': len(pred_nodes),
            'num_gold_edges': len(gold_edges),
            'num_pred_edges': len(pred_edges)
        }
print("✓ Judge 8 (Completeness) defined")

✓ Judge 8 (Completeness) defined


In [ ]:
class Judge9_GraphEditDistance:
    """Judge 9: Graph Edit Distance - Measures minimum edit operations to transform pred to gold."""

    def __init__(self):
        self.name = "graph_edit_distance"

    def _normalize_node(self, node: dict) -> tuple:
        """Convert node to normalized tuple for comparison."""
        node_id = str(node.get('id', ''))
        node_type = str(node.get('node_type', node.get('type', ''))).upper()

        attr = ''
        attrs = node.get('attrs', {})
        for key in ['name', 'text', 'title', 'term']:
            if key in node and node[key]:
                attr = normalize_text(str(node[key]))
                break
            elif isinstance(attrs, dict) and key in attrs and attrs[key]:
                attr = normalize_text(str(attrs[key]))
                break

        return (node_id, node_type, attr)

    def _normalize_edge(self, edge: dict) -> tuple:
        """Convert edge to normalized tuple for comparison."""
        src = str(edge.get('src', edge.get('source', edge.get('from', ''))))
        tgt = str(edge.get('tgt', edge.get('target', edge.get('to', ''))))
        edge_type = str(edge.get('type', edge.get('edge_type', edge.get('label', '')))).upper()

        return (src, tgt, edge_type)

    def _calculate_node_distance(self, node1: tuple, node2: tuple) -> int:
        """Calculate edit distance between two nodes."""
        id1, type1, attr1 = node1
        id2, type2, attr2 = node2

        distance = 0

        if id1 != id2:
            distance += 1

        if type1 != type2:
            distance += 1

        if attr1 != attr2:
            distance += 1

        return distance

    def _approximate_ged(self, pred_nodes: list, gold_nodes: list,
                         pred_edges: list, gold_edges: list) -> Dict[str, Any]:
        """Calculate approximate Graph Edit Distance using greedy matching."""

        pred_nodes_norm = [self._normalize_node(n) for n in pred_nodes]
        gold_nodes_norm = [self._normalize_node(n) for n in gold_nodes]
        pred_edges_norm = [self._normalize_edge(e) for e in pred_edges]
        gold_edges_norm = [self._normalize_edge(e) for e in gold_edges]

        pred_id_map = {n[0]: n for n in pred_nodes_norm}
        gold_id_map = {n[0]: n for n in gold_nodes_norm}

        node_edits = 0
        matched_gold_nodes = set()

        for pred_node in pred_nodes_norm:
            best_match = None
            best_distance = float('inf')

            for i, gold_node in enumerate(gold_nodes_norm):
                if i in matched_gold_nodes:
                    continue

                distance = self._calculate_node_distance(pred_node, gold_node)

                if distance < best_distance:
                    best_distance = distance
                    best_match = i

            if best_match is not None:
                matched_gold_nodes.add(best_match)
                node_edits += best_distance
            else:
                node_edits += 3

        node_insertions = len(gold_nodes_norm) - len(matched_gold_nodes)
        node_edits += node_insertions * 3

        pred_edge_set = set(pred_edges_norm)
        gold_edge_set = set(gold_edges_norm)

        edge_deletions = len(pred_edge_set - gold_edge_set)
        edge_insertions = len(gold_edge_set - pred_edge_set)
        edge_edits = edge_deletions + edge_insertions

        total_ged = node_edits + edge_edits

        max_nodes = max(len(pred_nodes_norm), len(gold_nodes_norm), 1)
        max_edges = max(len(pred_edges_norm), len(gold_edges_norm), 1)
        max_distance = (max_nodes * 3) + max_edges

        normalized_distance = min(1.0, total_ged / max_distance) if max_distance > 0 else 0.0

        similarity_score = max(0.0, min(1.0, 1.0 - normalized_distance))

        return {
            'ged': total_ged,
            'node_edits': node_edits,
            'edge_edits': edge_edits,
            'node_insertions': node_insertions,
            'node_deletions': len(pred_nodes_norm) - len(matched_gold_nodes),
            'edge_insertions': edge_insertions,
            'edge_deletions': edge_deletions,
            'normalized_distance': normalized_distance,
            'similarity_score': similarity_score
        }

    def score(self, pred: dict, gold: dict) -> Dict[str, Any]:
        pred_json = safe_json_parse(pred) if isinstance(pred, str) else pred
        gold_json = safe_json_parse(gold) if isinstance(gold, str) else gold

        if pred_json is None or gold_json is None:
            return {
                'score': 0.0,
                'ged': None,
                'valid_json': pred_json is not None
            }

        pred_json = normalize_graph_schema(pred_json)
        gold_json = normalize_graph_schema(gold_json)

        pred_nodes = extract_nodes(pred_json)
        gold_nodes = extract_nodes(gold_json)
        pred_edges = extract_edges(pred_json)
        gold_edges = extract_edges(gold_json)

        ged_result = self._approximate_ged(pred_nodes, gold_nodes, pred_edges, gold_edges)

        return {
            'score': ged_result['similarity_score'],
            'ged': ged_result['ged'],
            'node_edits': ged_result['node_edits'],
            'edge_edits': ged_result['edge_edits'],
            'node_insertions': ged_result['node_insertions'],
            'node_deletions': ged_result['node_deletions'],
            'edge_insertions': ged_result['edge_insertions'],
            'edge_deletions': ged_result['edge_deletions'],
            'normalized_distance': ged_result['normalized_distance'],
            'valid_json': True
        }


print("✓ Judge 9 (Graph Edit Distance) defined")

✓ Judge 9 (Graph Edit Distance) defined


In [ ]:
#Configs for no‑gold judges
!pip -q install google-generativeai tenacity
from dataclasses import dataclass
from typing import Dict, Any, Optional
import json, os, re
from tenacity import retry, stop_after_attempt, wait_exponential, retry_if_exception_type
from collections import defaultdict
from typing import Dict, Any, List, Optional

GOOGLE_API_KEY = os.getenv("gemini")
USE_GEMINI_NO_GOLD = bool(GOOGLE_API_KEY)
print(f"Gemini no‑gold judge enabled? {USE_GEMINI_NO_GOLD}")

Gemini no‑gold judge enabled? False


In [ ]:
# Non Gold Feature Judges and helpers
import json
import re
from collections import defaultdict
from typing import Dict, Any, List, Optional

def _safe_json_parse(x: Any) -> Optional[dict]:
    if isinstance(x, dict):
        return x
    if not isinstance(x, str):
        return None
    try:
        return json.loads(x)
    except Exception:
        return None

def _normalize_term(label: str) -> str:
    if label is None:
        return ""
    t = re.sub(r'[\"“”]', "", str(label))
    return t.strip().lower()

def _extract_nodes(g: dict) -> List[dict]:
    return list(g.get("nodes", []))

def _extract_edges(g: dict) -> List[dict]:
    return list(g.get("edges", []))

def _normalize_graph_schema(g: dict) -> dict:
    return g or {}


class BaseJudgeNoGold:
    def score(self, pred: dict, gold: dict = None) -> Dict[str, Any]:
        pred_json = _safe_json_parse(pred) if isinstance(pred, str) else pred
        if pred_json is None or not isinstance(pred_json, dict):
            return {"score": 0.0, "valid_json": False}
        pred_json = _normalize_graph_schema(pred_json)
        nodes = _extract_nodes(pred_json)
        edges = _extract_edges(pred_json)
        node_type = {str(n.get("id")): str(n.get("node_type", n.get("type", ""))).upper() for n in nodes}
        node_label = {str(n.get("id")): str(n.get("label", n.get("name", ""))) for n in nodes}
        return self._compute_score(nodes, edges, node_type, node_label)

    def _compute_score(self, nodes, edges, node_type, node_label):
        raise NotImplementedError

class JudgeUnusedDefinitionsNoGold(BaseJudgeNoGold):
    def __init__(self):
        self.name = "unused_definitions"

    def _compute_score(self, nodes, edges, node_type, node_label):
        defined_term_ids = [nid for nid, t in node_type.items() if t == "DEFINED_TERM"]
        defining_edges = [e for e in edges if e.get("type") == "DEFINES"]
        defined_by_clause = {str(e.get("tgt")) for e in defining_edges}
        unused = [nid for nid in defined_term_ids if nid not in defined_by_clause]
        total = len(defined_term_ids)
        score = 1.0 - (len(unused) / max(1, total))
        return {
            "score": score,
            "unused_term_ids": unused,
            "defined_term_ids": defined_term_ids,
            "valid_json": True,
        }

class JudgeUndefinedReferencesNoGold(BaseJudgeNoGold):
    def __init__(self):
        self.name = "undefined_references"

    def _compute_score(self, nodes, edges, node_type, node_label):
        clause_node_ids = [nid for nid, t in node_type.items() if t == "CLAUSE"]
        text_labels = {nid: _normalize_term(label) for nid, label in node_label.items() if nid in clause_node_ids}
        refers_edges = [e for e in edges if e.get("type") == "REFERS_TO"]
        targets = {str(e.get("tgt")) for e in refers_edges}
        referenced_labels = {_normalize_term(node_label.get(tgt, "")) for tgt in targets}

        # check if any referenced label is not actually mentioned among current clause nodes
        missing = [t for t in targets if _normalize_term(node_label.get(t, "")) not in text_labels.values()]
        score = 1.0 - (len(missing) / max(1, len(targets)))
        return {
            "score": score,
            "invalid_targets": missing,
            "valid_json": True,
        }

class JudgeDuplicateDefinitionsNoGold(BaseJudgeNoGold):
    def __init__(self):
        self.name = "duplicate_definitions"

    def _compute_score(self, nodes, edges, node_type, node_label):
        term_map = defaultdict(list)
        for nid, t in node_type.items():
            if t in {"DEFINED_TERM", "VALUE"}:
                norm = _normalize_term(node_label.get(nid, ""))
                for existing in term_map:
                    if norm in existing or existing in norm:
                        term_map[existing].append(nid)
                        break
                else:
                    term_map[norm].append(nid)
        dup_groups = {k: v for k, v in term_map.items() if len(v) > 1}
        total_terms = sum(1 for t in node_type.values() if t in {"DEFINED_TERM", "VALUE"})
        score = 1.0 - (len(dup_groups) / max(1, total_terms))
        return {
            "score": score,
            "duplicate_groups": dup_groups,
            "total_terms": total_terms,
            "valid_json": True,
        }

class JudgeOrphanNodesNoGold(BaseJudgeNoGold):
    def __init__(self):
        self.name = "orphan_nodes"

    def _compute_score(self, nodes, edges, node_type, node_label):
        connected = set()
        for e in edges:
            connected.add(str(e.get("src") or e.get("from")))
            connected.add(str(e.get("tgt") or e.get("to")))
        all_ids = set(node_type.keys())
        orphans = list(all_ids - connected)
        score = 1.0 - (len(orphans) / max(1, len(all_ids)))
        return {
            "score": score,
            "num_orphans": len(orphans),
            "orphan_node_ids": orphans,
            "valid_json": True,
        }

class JudgeIsolatedComponentsNoGold(BaseJudgeNoGold):
    def __init__(self):
        self.name = "isolated_components"

    def _compute_score(self, nodes, edges, node_type, node_label):
        adj = defaultdict(set)
        for e in edges:
            src = str(e.get("src") or e.get("from"))
            tgt = str(e.get("tgt") or e.get("to"))
            adj[src].add(tgt)
            adj[tgt].add(src)

        visited = set()
        components = []

        def dfs(u, comp):
            visited.add(u)
            comp.append(u)
            for v in adj[u]:
                if v not in visited:
                    dfs(v, comp)

        for nid in node_type:
            if nid not in visited:
                comp = []
                dfs(nid, comp)
                components.append(comp)

        if len(components) <= 1:
            return {"score": 1.0, "num_isolated_components": 0, "valid_json": True}

        isolated = [c for c in components[1:] if len(c) <= 8]
        score = 1.0 - (len(isolated) / max(1, len(components) - 1))
        return {
            "score": score,
            "num_isolated_components": len(isolated),
            "total_components": len(components),
            "valid_json": True,
        }


In [ ]:
class JudgeOntologyComplianceNoGold(BaseJudgeNoGold):
    """Check compliance of a predicted minigraph against ontology rules.

    This judge verifies that:
      * Every edge's source and target node types conform to the allowed
        ontology edge rules.
      * Clause nodes with hierarchical IDs (e.g., "3.4") have an
        inferred parent clause node present in the graph.  The parent is
        inferred by taking the substring before the last dot (".") and
        optionally checking for a matching "ARTICLE <number>" node.
      * For each clause that has a parent, there must be an IS_PART_OF
        edge from the child clause to its parent clause.

    The compliance score is computed as:
        score = 1 - (num_invalid_edges + num_missing_parents + num_missing_edges)
                     / max(1, total_edges + 2 * num_clause_with_parent)
    """

    def __init__(self) -> None:
        self.name = 'ontology_compliance'

    # Allowed (src_type, tgt_type) for each edge type
    _ALLOWED: Dict[str, Tuple[str, str]] = {
        'IS_PART_OF': ('CLAUSE', 'CLAUSE'),
        'DEFINES': ('CLAUSE', 'DEFINED_TERM'),
        'USES': ('CLAUSE', 'DEFINED_TERM'),
        'REFERENCES': ('CLAUSE', 'CLAUSE'),
        'MENTIONS_PARTY': ('CLAUSE', 'PARTY'),
        'CONTAINS': ('CLAUSE', 'VALUE'),
    }

    def _infer_parent_ids(self, clause_id: str, node_type_map: Dict[str, str]) -> List[str]:
        """Return plausible parent IDs for a clause."""
        if '.' not in clause_id:
            return []
        prefix = clause_id.rsplit('.', 1)[0]
        parents = []
        if prefix in node_type_map and node_type_map[prefix] == 'CLAUSE':
            parents.append(prefix)
        art_prefix = f'ARTICLE {prefix}'
        if art_prefix in node_type_map and node_type_map[art_prefix] == 'CLAUSE':
            parents.append(art_prefix)
        return parents

    def _compute_score(self, nodes, edges, node_type_map, node_label_map):
        invalid_edges = []
        missing_parents = []
        missing_is_part_of = []

        for e in edges:
            etype = str(e.get('type', e.get('edge_type', ''))).upper()
            src_id = str(e.get('src') or e.get('source') or e.get('from'))
            tgt_id = str(e.get('tgt') or e.get('target') or e.get('to'))
            src_type = node_type_map.get(src_id)
            tgt_type = node_type_map.get(tgt_id)
            allowed_pair = self._ALLOWED.get(etype)
            if src_type is None or tgt_type is None or allowed_pair is None or (src_type, tgt_type) != allowed_pair:
                invalid_edges.append((etype, src_id, tgt_id))

        is_part_of_edges = set()
        for e in edges:
            if str(e.get('type', e.get('edge_type', ''))).upper() == 'IS_PART_OF':
                child = str(e.get('src') or e.get('source') or e.get('from'))
                parent = str(e.get('tgt') or e.get('target') or e.get('to'))
                is_part_of_edges.add((child, parent))

        clause_with_parent = 0
        for node_id, ntype in node_type_map.items():
            if ntype != 'CLAUSE' or '.' not in node_id:
                continue
            clause_with_parent += 1
            parent_candidates = self._infer_parent_ids(node_id, node_type_map)
            if not parent_candidates:
                missing_parents.append(node_id)
                continue
            if not any((node_id, parent) in is_part_of_edges for parent in parent_candidates):
                missing_is_part_of.append((node_id, parent_candidates[0]))

        total_edges = len(edges)
        total_clauses = clause_with_parent
        total_expected = total_edges + 2 * total_clauses
        issues = len(invalid_edges) + len(missing_parents) + len(missing_is_part_of)
        score = 1.0 - (issues / max(1, total_expected))
        return {
            'score': max(0.0, min(1.0, score)),
            'num_invalid_edges': len(invalid_edges),
            'num_missing_parents': len(missing_parents),
            'num_missing_is_part_of_edges': len(missing_is_part_of),
            'invalid_edges': invalid_edges,
            'missing_parent_clauses': missing_parents,
            'missing_is_part_of_pairs': missing_is_part_of,
            'total_edges': total_edges,
            'clauses_with_parent': total_clauses,
            'valid_json': True,
        }

class JudgeDomainRangeValidityNoGold(BaseJudgeNoGold):
    """Check if edge connects valid node type pairs as per ontology rules."""
    def __init__(self):
        self.name = "domain_range_validity"
        self.allowed_pairs = {
            ("CLAUSE", "DEFINED_TERM"): {"DEFINES", "USES"},
            ("CLAUSE", "VALUE"): {"CONTAINS"},
            ("CLAUSE", "PARTY"): {"MENTIONS_PARTY"},
            ("CLAUSE", "CLAUSE"): {"REFERENCES, IS_PART_OF"},
        }

    def _compute_score(self, nodes, edges, node_type, node_label):
        invalid_edges = []
        for e in edges:
            etype = e.get("type")
            src = str(e.get("src") or e.get("from"))
            tgt = str(e.get("tgt") or e.get("to"))
            src_type = node_type.get(src)
            tgt_type = node_type.get(tgt)
            if not src_type or not tgt_type:
                continue
            if (src_type, tgt_type) not in self.allowed_pairs:
                invalid_edges.append(e)
            elif etype not in self.allowed_pairs[(src_type, tgt_type)]:
                invalid_edges.append(e)
        score = 1.0 - len(invalid_edges) / max(1, len(edges))
        return {
            "score": score,
            "num_invalid_edges": len(invalid_edges),
            "invalid_edges": invalid_edges,
            "valid_json": True,
        }

class JudgeLabelAccuracyNoGold(BaseJudgeNoGold):
    """Check if labels on CLAUSE or DEFINED_TERM nodes appear in the input clause text."""
    def __init__(self):
        self.name = "label_accuracy"

    def _compute_score(self, nodes, edges, node_type, node_label, clause_text: str):
        self.clause_text = clause_text.lower()
        relevant_ids = [nid for nid, t in node_type.items() if t in {"CLAUSE", "DEFINED_TERM"}]
        mismatches = [nid for nid in relevant_ids if _normalize_term(node_label.get(nid, "")) not in self.clause_text]
        score = 1.0 - len(mismatches) / max(1, len(relevant_ids))
        return {
            "score": score,
            "unmatched_node_ids": mismatches,
            "valid_json": True,
        }

## 6. Initialize All Judges

In [ ]:
# Initialize all judges
print("Initializing judges...")
import google.generativeai as genai
from google.colab import userdata
import openai


try:
    OPENAI_API_KEY = userdata.get('OpenAIService')
    if OPENAI_API_KEY:
        openai.api_key = OPENAI_API_KEY
        USE_OPENAI = True
        print("✓ OpenAI API key loaded")
    else:
        USE_OPENAI = False
        print("⚠ OpenAI API key not found - judge will be disabled")
except:
    USE_OPENAI = False
    print("⚠ OpenAI API key not available - judge will be disabled")


# Initialize judges
judge1 = Judge1_ExactMatch()
judge2 = Judge2_StructuralF1()
judge3 = Judge3_SemanticSimilarity(embedding_model)
judge4 = Judge4_OpenAI_Issues(enabled=USE_OPENAI)
judge6 = Judge6_Hybrid(judge2, judge3, alpha=0.5)  # Equal weighting
judge7 = Judge7_ParseSuccess()
judge8 = Judge8_Completeness()
judge9 = Judge9_GraphEditDistance()

# New no‑gold linter judges
judge_unused_defs       = JudgeUnusedDefinitionsNoGold()
judge_undefined_refs    = JudgeUndefinedReferencesNoGold()
judge_duplicate_defs    = JudgeDuplicateDefinitionsNoGold()
judge_orphan_nodes      = JudgeOrphanNodesNoGold()
judge_isolated_comps    = JudgeIsolatedComponentsNoGold()
judge_label_accuracy = JudgeLabelAccuracyNoGold()
judge_domain_range_validity = JudgeDomainRangeValidityNoGold()
judge_ontology_compliance = JudgeOntologyComplianceNoGold()


# Combine all judges
all_judges = [
    judge1, judge2, judge3, judge4, judge6, judge7, judge8, judge9,
    judge_unused_defs, judge_undefined_refs,
    judge_duplicate_defs, judge_orphan_nodes,
    judge_isolated_comps, judge_label_accuracy,
    judge_domain_range_validity, judge_ontology_compliance
]

print(f"\n✓ Initialized {len(all_judges)} judges")
for judge in all_judges:
    enabled_status = ""
    if judge.name == "openai_issues":
        enabled_status = " (ENABLED)" if USE_OPENAI else " (DISABLED - no API key)"
    elif judge.name == "gemini_changes":
        enabled_status = " (ENABLED)" if USE_GEMINI else " (DISABLED - no API key)"
    elif judge.name == "gemini_nogold":
        enabled_status = " (ENABLED)" if USE_GEMINI_NO_GOLD else " (DISABLED - no API key)"
    print(f"  - {judge.name}{enabled_status}")




Initializing judges...
✓ OpenAI API key loaded
✓ Gemini API key loaded
✓ openai_issues initialized with API access (model: gpt-5-thinking, timeout=60.0s)

✓ Initialized 17 judges
  - exact_match
  - structural_f1
  - semantic_similarity
  - openai_issues (ENABLED)
  - hybrid
  - parse_success
  - completeness
  - graph_edit_distance
  - unused_definitions
  - undefined_references
  - duplicate_definitions
  - orphan_nodes
  - isolated_components
  - ontology_compliance
  - label_accuracy
  - domain_range_validity
  - ontology_compliance


## 7. Load Predictions

In [ ]:
# Load predictions from JSONL file
predictions_file = PREDICTIONS_DIR / "predictions.jsonl"

print(f"Loading predictions from {predictions_file}...")

predictions = []
if predictions_file.exists():
    with open(predictions_file, 'r') as f:
        for line in f:
            if line.strip():
                predictions.append(json.loads(line))
else:
    # Load from individual sample files
    print("JSONL not found, loading from samples directory...")
    sample_files = sorted(SAMPLES_DIR.glob("idx_*.json"))
    for sample_file in sample_files:
        with open(sample_file, 'r') as f:
            predictions.append(json.load(f))

print(f"✓ Loaded {len(predictions)} predictions")

# Show sample
if predictions:
    print("\nSample prediction keys:")
    print(list(predictions[0].keys()))

Loading predictions from /content/drive/MyDrive/multiJudgePredictions-full/predictions.jsonl...
✓ Loaded 150 predictions

Sample prediction keys:
['index', 'id', 'contract_id', 'clause_id', 'prompt', 'gold_minigraph', 'prediction_text', 'prediction_json', 'prediction_parse_note']


## 8. Run Evaluationss

In [ ]:
# Run all judges on all predictions
print("Running evaluation...")

results = []

for pred_record in tqdm(predictions, desc="Evaluating"):
    # Extract data
    sample_id = pred_record.get('id', pred_record.get('index', 'unknown'))
    contract_id = pred_record.get('contract_id', '')
    clause_id = pred_record.get('clause_id', '')

    # Get prediction and gold
    pred_text = pred_record.get('prediction_text', '')
    gold_text = pred_record.get('gold_minigraph', '')

    # Get original prompt and clause text
    prompt_data = pred_record.get('prompt', '')
    original_prompt = ''
    clause_text = ''

    # Try to extract clause text and instruction from prompt
    if prompt_data:
        try:
            prompt_json = safe_json_parse(prompt_data)
            if prompt_json:
                original_prompt = prompt_json.get('instruction', '')
                input_data = prompt_json.get('input', {})
                if isinstance(input_data, dict):
                    clause_info = input_data.get('clause', {})
                    if isinstance(clause_info, dict):
                        clause_text = clause_info.get('text', '')
        except:
            pass

    # Parse JSONs
    pred_json = pred_record.get('prediction_json')
    if pred_json is None:
        pred_json = safe_json_parse(pred_text)

    gold_json = safe_json_parse(gold_text)

    # Run all judges
    judge_scores = {}

    for judge in all_judges:
        try:
            # Check if judge needs clause context (LLM judges)
            if judge.name in ['openai_issues', 'gemini_changes', 'judge_label_accuracy']:
                score_result = judge.score(
                    pred_json or pred_text,
                    gold_json or gold_text,
                    clause_text=clause_text,
                    original_prompt=original_prompt
                )
            else:
                score_result = judge.score(pred_json or pred_text, gold_json or gold_text)

            judge_scores[judge.name] = score_result
        except Exception as e:
            print(f"\nError in {judge.name} for sample {sample_id}: {e}")
            judge_scores[judge.name] = {
                'score': None,
                'error': str(e)
            }

    # Compile result

    result = {
        'sample_id': sample_id,
        'contract_id': contract_id,
        'clause_id': clause_id,
        'judges': judge_scores,
        'prediction_text': pred_text,
        'gold_text': gold_text,
        'clause_text': clause_text,
        'original_prompt': original_prompt,
        'valid_prediction': pred_json is not None,
        'valid_gold': gold_json is not None
    }

    results.append(result)

print(f"\n✓ Evaluation complete: {len(results)} samples processed")

Running evaluation...


Evaluating:   0%|          | 0/150 [00:00<?, ?it/s]

Streaming output truncated to the last 5000 lines.
 
NODE RULES
 
- CLAUSE:
  { "id": "<clause-id>", "node_type": "CLAUSE", "title": "<title>", "level": <int> }
  Rule: Create a node for the input clause, its inferred parent, and any clauses it explicitly references.

- DEFINED_TERM:
  { "id": "term:<Canonical Term>", "node_type": "DEFINED_TERM", "name": "<Canonical Term>" }
  Rule: Create for terms with special meaning (quoted, ALL CAPS, or explicitly defined). Canonicalize the name (e.g., "this Agreement" becomes "Agreement"; use Title Case).

- PARTY:
  { "id": "party:<Party Name>", "node_type": "PARTY", "name": "<Party Name>" }
  Rule: Only for legal entities or defined roles (e.g., "Licensor", "the Supplier"). Canonicalize by removing articles ("the", "a").

- VALUE:
  { "id": "value:<literal text>", "node_type": "VALUE", "unit": "Currency|Percentage|Days|Months|Years", "text": "<literal text>" }
  Rule: Extract specific amounts, durations, etc.

 
EDGE RULES
 
Format: { "src":"<i

## 9. Save Results

In [ ]:
# Save detailed results as JSONL
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
results_file = OUTPUT_DIR / f"judge_scores_{timestamp}.jsonl"

print(f"Saving detailed results to {results_file}...")

with open(results_file, 'w') as f:
    for result in results:
        f.write(json.dumps(result, ensure_ascii=False) + '\n')

print(f"✓ Saved detailed results")

Saving detailed results to /content/drive/MyDrive/multiJudgeScores_non_gold/judge_scores_20251104_201357.jsonl...
✓ Saved detailed results


In [ ]:
# Create summary CSV
summary_file = OUTPUT_DIR / f"judge_scores_summary_{timestamp}_fixed.csv"

print(f"Creating summary CSV at {summary_file}...")

summary_data = []
for result in results:
    row = {
        'sample_id': result['sample_id'],
        'contract_id': result['contract_id'],
        'clause_id': result['clause_id'],
        'valid_prediction': result['valid_prediction'] if isinstance(result['valid_prediction'], bool) else result['valid_prediction'] == 'True',
    }

    # Add main scores from each judge
    for judge_name, judge_result in result['judges'].items():
        row[f"{judge_name}_score"] = judge_result.get('score')

    summary_data.append(row)

df_summary = pd.DataFrame(summary_data)

# Ensure valid_prediction is boolean type
df_summary['valid_prediction'] = df_summary['valid_prediction'].astype(bool)

df_summary.to_csv(summary_file, index=False)

print(f"✓ Saved summary CSV")
print(f"\nSummary shape: {df_summary.shape}")
print(f"Data types:")
print(df_summary.dtypes)
print(f"\nFirst few rows:")
display(df_summary.head())

Creating summary CSV at /content/drive/MyDrive/multiJudgeScores_non_gold/judge_scores_summary_20251104_201357_fixed.csv...
✓ Saved summary CSV

Summary shape: (150, 20)
Data types:
sample_id                       object
contract_id                     object
clause_id                       object
valid_prediction                  bool
exact_match_score              float64
structural_f1_score            float64
semantic_similarity_score      float64
openai_issues_score            float64
hybrid_score                   float64
parse_success_score            float64
completeness_score             float64
graph_edit_distance_score      float64
unused_definitions_score       float64
undefined_references_score     float64
duplicate_definitions_score    float64
orphan_nodes_score             float64
isolated_components_score      float64
ontology_compliance_score      float64
label_accuracy_score           float64
domain_range_validity_score    float64
dtype: object

First few rows:


,sample_id,contract_id,clause_id,valid_prediction,exact_match_score,structural_f1_score,semantic_similarity_score,openai_issues_score,hybrid_score,parse_success_score,completeness_score,graph_edit_distance_score,unused_definitions_score,undefined_references_score,duplicate_definitions_score,orphan_nodes_score,isolated_components_score,ontology_compliance_score,label_accuracy_score,domain_range_validity_score
0,NEOMEDIATECHNOLOGIESINC_12_15_2005-EX-16.1-DIS...,NEOMEDIATECHNOLOGIESINC_12_15_2005-EX-16.1-DIS...,18.2,True,0.0,0.690476,0.998083,0.95,0.844280,1.0,1.0,0.925926,0.0,1.0,0.750000,1.0,1.0,0.692308,NaN,0.833333
1,INTERNATIONALFASTFOODCORP_04_04_1997-EX-99-FRA...,INTERNATIONALFASTFOODCORP_04_04_1997-EX-99-FRA...,15.5.2.3,True,0.0,0.916084,0.938784,0.90,0.927434,1.0,1.0,0.851852,0.0,1.0,0.666667,1.0,1.0,0.769231,NaN,0.833333
2,INTERNATIONALFASTFOODCORP_04_04_1997-EX-99-FRA...,INTERNATIONALFASTFOODCORP_04_04_1997-EX-99-FRA...,5.3,True,0.0,0.250000,0.978952,0.97,0.614476,1.0,1.0,0.857143,1.0,1.0,1.000000,1.0,1.0,0.666667,NaN,0.000000
3,INTERNATIONALFASTFOODCORP_04_04_1997-EX-99-FRA...,INTERNATIONALFASTFOODCORP_04_04_1997-EX-99-FRA...,1.9,True,0.0,0.550000,0.907238,0.50,0.728619,1.0,0.7,0.578947,0.0,1.0,1.000000,1.0,1.0,0.666667,NaN,0.500000
4,INTERNATIONALFASTFOODCORP_04_04_1997-EX-99-FRA...,INTERNATIONALFASTFOODCORP_04_04_1997-EX-99-FRA...,11.1.1,True,1.0,1.000000,1.000000,1.00,1.000000,1.0,1.0,1.000000,0.0,1.0,1.000000,1.0,1.0,0.666667,NaN,0.750000


In [ ]:
df_summary.describe()

,exact_match_score,structural_f1_score,semantic_similarity_score,openai_issues_score,hybrid_score,parse_success_score,completeness_score,graph_edit_distance_score,unused_definitions_score,undefined_references_score,duplicate_definitions_score,orphan_nodes_score,isolated_components_score,ontology_compliance_score,label_accuracy_score,domain_range_validity_score
count,150.000000,150.000000,150.000000,150.000000,150.000000,150.000000,150.000000,150.000000,150.000000,150.000000,150.000000,150.000000,150.000000,150.000000,1.0,150.000000
mean,0.233333,0.760881,0.913183,0.732267,0.837032,0.985333,0.847271,0.696805,0.272444,0.993333,0.940159,0.971987,0.980000,0.728652,0.0,0.742775
std,0.424370,0.223399,0.128145,0.237125,0.157413,0.098571,0.178719,0.256179,0.431043,0.081650,0.156214,0.161865,0.140469,0.119911,NaN,0.282376
min,0.000000,0.000000,0.000000,0.200000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.000000
25%,0.000000,0.657193,0.872115,0.550000,0.750000,1.000000,0.752083,0.531100,0.000000,1.000000,1.000000,1.000000,1.000000,0.666667,0.0,0.666667
50%,0.000000,0.775000,0.953568,0.800000,0.857641,1.000000,0.900000,0.703704,0.000000,1.000000,1.000000,1.000000,1.000000,0.727273,0.0,0.800000
75%,0.000000,1.000000,1.000000,0.965000,1.000000,1.000000,1.000000,1.000000,0.625000,1.000000,1.000000,1.000000,1.000000,0.777778,0.0,1.000000
max,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,0.0,1.000000


In [ ]:
print("="*60)
print("POST-PROCESSING VALIDATION")
print("="*60)

# Validate all scores are in [0, 1] range
print("\n1. Checking score ranges...")
score_columns = [col for col in df_summary.columns if col.endswith('_score')]

violations = []
for col in score_columns:
    scores = df_summary[col].dropna()
    if len(scores) > 0:
        min_val = scores.min()
        max_val = scores.max()
        out_of_range = scores[(scores < 0) | (scores > 1)]

        if len(out_of_range) > 0 or min_val < 0 or max_val > 1:
            violations.append({
                'judge': col.replace('_score', ''),
                'min': min_val,
                'max': max_val,
                'violations': len(out_of_range)
            })
            print(f"  ⚠️ {col}: min={min_val:.4f}, max={max_val:.4f}, violations={len(out_of_range)}")
        else:
            print(f"  ✓ {col}: Range OK [{min_val:.4f}, {max_val:.4f}]")

if len(violations) == 0:
    print("\n✅ All scores are in valid [0, 1] range!")
else:
    print(f"\n❌ Found {len(violations)} judges with range violations")

# Validate data types
print("\n2. Checking data types...")
if df_summary['valid_prediction'].dtype == bool:
    print("  ✓ valid_prediction is boolean")
else:
    print(f"  ⚠️ valid_prediction is {df_summary['valid_prediction'].dtype} (should be boolean)")

# Check judge coverage
print("\n3. Checking judge coverage...")
for col in score_columns:
    non_null = df_summary[col].notna().sum()
    pct = (non_null / len(df_summary)) * 100
    status = "✓" if pct >= 90 else "⚠️"
    print(f"  {status} {col}: {non_null}/{len(df_summary)} ({pct:.1f}%)")

print("\n" + "="*60)
print("VALIDATION COMPLETE")
print("="*60)

POST-PROCESSING VALIDATION

1. Checking score ranges...
  ✓ exact_match_score: Range OK [0.0000, 1.0000]
  ✓ structural_f1_score: Range OK [0.0000, 1.0000]
  ⚠️ semantic_similarity_score: min=0.0000, max=1.0000, violations=21
  ✓ openai_issues_score: Range OK [0.2000, 1.0000]
  ⚠️ hybrid_score: min=0.0000, max=1.0000, violations=20
  ✓ parse_success_score: Range OK [0.0000, 1.0000]
  ✓ completeness_score: Range OK [0.0000, 1.0000]
  ✓ graph_edit_distance_score: Range OK [0.0000, 1.0000]
  ✓ unused_definitions_score: Range OK [0.0000, 1.0000]
  ✓ undefined_references_score: Range OK [0.0000, 1.0000]
  ✓ duplicate_definitions_score: Range OK [0.0000, 1.0000]
  ✓ orphan_nodes_score: Range OK [0.0000, 1.0000]
  ✓ isolated_components_score: Range OK [0.0000, 1.0000]
  ✓ ontology_compliance_score: Range OK [0.0000, 1.0000]
  ✓ label_accuracy_score: Range OK [0.0000, 0.0000]
  ✓ domain_range_validity_score: Range OK [0.0000, 1.0000]

❌ Found 2 judges with range violations

2. Checking data ty

In [ ]:
# Create detailed metrics CSV with all sub-scores
detailed_file = OUTPUT_DIR / f"judge_scores_detailed_{timestamp}.csv"

print(f"Creating detailed metrics CSV at {detailed_file}...")

detailed_data = []
for result in results:
    row = {
        'sample_id': result['sample_id'],
        'contract_id': result['contract_id'],
        'clause_id': result['clause_id'],
    }

    # Add all metrics from each judge
    for judge_name, judge_result in result['judges'].items():
        for key, value in judge_result.items():
            # Flatten the results
            col_name = f"{judge_name}_{key}"
            row[col_name] = value

    detailed_data.append(row)

df_detailed = pd.DataFrame(detailed_data)
df_detailed.to_csv(detailed_file, index=False)

print(f"✓ Saved detailed metrics CSV")
print(f"\nDetailed shape: {df_detailed.shape}")

Creating detailed metrics CSV at /content/drive/MyDrive/multiJudgeScores_non_gold/judge_scores_detailed_20251104_201357.csv...
✓ Saved detailed metrics CSV

Detailed shape: (150, 81)


## 10. Analysis and Statistics

In [ ]:
# Calculate aggregate statistics
print("\n" + "="*60)
print("AGGREGATE STATISTICS")
print("="*60)

score_columns = [col for col in df_summary.columns if col.endswith('_score')]

print(f"\nNumber of samples: {len(df_summary)}")
print(f"Valid predictions: {df_summary['valid_prediction'].sum()} ({df_summary['valid_prediction'].mean()*100:.1f}%)")

print("\n" + "-"*60)
print("JUDGE SCORES (mean ± std)")
print("-"*60)

for col in score_columns:
    judge_name = col.replace('_score', '')
    scores = df_summary[col].dropna()

    if len(scores) > 0:
        mean_score = scores.mean()
        std_score = scores.std()
        print(f"{judge_name:25s}: {mean_score:.4f} ± {std_score:.4f} (n={len(scores)})")
    else:
        print(f"{judge_name:25s}: No valid scores")


AGGREGATE STATISTICS

Number of samples: 150
Valid predictions: 149 (99.3%)

------------------------------------------------------------
JUDGE SCORES (mean ± std)
------------------------------------------------------------
exact_match              : 0.2333 ± 0.4244 (n=150)
structural_f1            : 0.7609 ± 0.2234 (n=150)
semantic_similarity      : 0.9132 ± 0.1281 (n=150)
openai_issues            : 0.7323 ± 0.2371 (n=150)
hybrid                   : 0.8370 ± 0.1574 (n=150)
parse_success            : 0.9853 ± 0.0986 (n=150)
completeness             : 0.8473 ± 0.1787 (n=150)
graph_edit_distance      : 0.6968 ± 0.2562 (n=150)
unused_definitions       : 0.2724 ± 0.4310 (n=150)
undefined_references     : 0.9933 ± 0.0816 (n=150)
duplicate_definitions    : 0.9402 ± 0.1562 (n=150)
orphan_nodes             : 0.9720 ± 0.1619 (n=150)
isolated_components      : 0.9800 ± 0.1405 (n=150)
ontology_compliance      : 0.7287 ± 0.1199 (n=150)
label_accuracy           : 0.0000 ± nan (n=1)
domain_range_v

In [ ]:
# Show per-judge statistics from detailed metrics
print("\n" + "="*60)
print("DETAILED METRICS")
print("="*60)

# Structural F1 details
if 'structural_f1_node_f1' in df_detailed.columns:
    print("\nStructural F1 Judge:")
    print(f"  Node F1:      {df_detailed['structural_f1_node_f1'].mean():.4f}")
    print(f"  Node Prec:    {df_detailed['structural_f1_node_precision'].mean():.4f}")
    print(f"  Node Recall:  {df_detailed['structural_f1_node_recall'].mean():.4f}")
    print(f"  Edge F1:      {df_detailed['structural_f1_edge_f1'].mean():.4f}")
    print(f"  Edge Prec:    {df_detailed['structural_f1_edge_precision'].mean():.4f}")
    print(f"  Edge Recall:  {df_detailed['structural_f1_edge_recall'].mean():.4f}")

# Completeness details
if 'completeness_node_coverage' in df_detailed.columns:
    print("\nCompleteness Judge:")
    print(f"  Node Coverage: {df_detailed['completeness_node_coverage'].mean():.4f}")
    print(f"  Edge Coverage: {df_detailed['completeness_edge_coverage'].mean():.4f}")
    print(f"  Type Coverage: {df_detailed['completeness_type_coverage'].mean():.4f}")

# Parse success details
if 'parse_success_valid_json' in df_detailed.columns:
    print("\nParse Success Judge:")
    valid_json_rate = df_detailed['parse_success_valid_json'].mean()
    has_nodes_rate = df_detailed['parse_success_has_nodes'].mean()
    has_edges_rate = df_detailed['parse_success_has_edges'].mean()
    print(f"  Valid JSON:       {valid_json_rate:.1%}")
    print(f"  Has Nodes:        {has_nodes_rate:.1%}")
    print(f"  Has Edges:        {has_edges_rate:.1%}")

# GED details
if 'graph_edit_distance_ged' in df_detailed.columns:
    print("\nGraph Edit Distance Judge:")
    avg_ged = df_detailed['graph_edit_distance_ged'].mean()
    avg_node_edits = df_detailed['graph_edit_distance_node_edits'].mean()
    avg_edge_edits = df_detailed['graph_edit_distance_edge_edits'].mean()
    avg_norm_dist = df_detailed['graph_edit_distance_normalized_distance'].mean()
    print(f"  Average GED:           {avg_ged:.2f}")
    print(f"  Average Node Edits:    {avg_node_edits:.2f}")
    print(f"  Average Edge Edits:    {avg_edge_edits:.2f}")
    print(f"  Normalized Distance:   {avg_norm_dist:.4f}")
    print(f"  Node Insertions:       {df_detailed['graph_edit_distance_node_insertions'].mean():.2f}")
    print(f"  Node Deletions:        {df_detailed['graph_edit_distance_node_deletions'].mean():.2f}")
    print(f"  Edge Insertions:       {df_detailed['graph_edit_distance_edge_insertions'].mean():.2f}")
    print(f"  Edge Deletions:        {df_detailed['graph_edit_distance_edge_deletions'].mean():.2f}")

# OpenAI Issues Judge details
if 'openai_issues_issues_found' in df_detailed.columns:
    print("\nOpenAI Issues Judge:")
    issues_data = df_detailed['openai_issues_issues_found'].dropna()
    if len(issues_data) > 0:
        print(f"  Average Issues Found:  {issues_data.mean():.2f}")
        print(f"  Min Issues:            {issues_data.min():.0f}")
        print(f"  Max Issues:            {issues_data.max():.0f}")
        print(f"  Samples Evaluated:     {len(issues_data)}")



DETAILED METRICS

Structural F1 Judge:
  Node F1:      0.8143
  Node Prec:    0.8147
  Node Recall:  0.8336
  Edge F1:      0.7075
  Edge Prec:    0.7064
  Edge Recall:  0.7435

Completeness Judge:
  Node Coverage: 0.8036
  Edge Coverage: 0.8579
  Type Coverage: 0.9406

Parse Success Judge:
  Valid JSON:       99.3%
  Has Nodes:        99.3%
  Has Edges:        97.3%

Graph Edit Distance Judge:
  Average GED:           6.23
  Average Node Edits:    4.01
  Average Edge Edits:    2.22
  Normalized Distance:   0.2985
  Node Insertions:       0.24
  Node Deletions:        0.44
  Edge Insertions:       0.97
  Edge Deletions:        1.26

OpenAI Issues Judge:
  Average Issues Found:  3.54
  Min Issues:            0
  Max Issues:            14
  Samples Evaluated:     150


In [ ]:
# Save statistics summary
stats_file = OUTPUT_DIR / f"statistics_summary_{timestamp}.txt"

print(f"\nSaving statistics to {stats_file}...")

with open(stats_file, 'w') as f:
    f.write("MULTI-JUDGE EVALUATION STATISTICS\n")
    f.write("="*60 + "\n\n")
    f.write(f"Evaluation Date: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
    f.write(f"Number of Samples: {len(df_summary)}\n")
    f.write(f"Valid Predictions: {df_summary['valid_prediction'].sum()} ({df_summary['valid_prediction'].mean()*100:.1f}%)\n")
    f.write("\n" + "-"*60 + "\n")
    f.write("JUDGE SCORES (mean ± std)\n")
    f.write("-"*60 + "\n\n")

    for col in score_columns:
        judge_name = col.replace('_score', '')
        scores = df_summary[col].dropna()

        if len(scores) > 0:
            mean_score = scores.mean()
            std_score = scores.std()
            f.write(f"{judge_name:25s}: {mean_score:.4f} ± {std_score:.4f} (n={len(scores)})\n")
        else:
            f.write(f"{judge_name:25s}: No valid scores\n")

print("✓ Saved statistics summary")


Saving statistics to /content/drive/MyDrive/multiJudgeScores_non_gold/statistics_summary_20251104_201357.txt...
✓ Saved statistics summary
